# Preprocesamiento

## Importación de librerias

In [55]:
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd

try:
    pa.unregister_extension_type("pandas.period")
except:
    pass

import numpy as np
import glob
from datetime import timedelta

In [48]:
import pyarrow
import fastparquet

## Importación de datos de archivos

Se tienen 30 archivos con datos crudos de medidores inteligentes. Cada archivo contiene un día de multiples medidores.

In [18]:
path = "C:\\Users\\rroman\\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\\Documents\\Unidad\\2026\\Capacitación\\Data Science y Machine Learning\\proyecto final\\data\\raw\\intervals_0000_*.txt"
files = glob.glob(path)

print("Número de archivos:", len(files))

Número de archivos: 30


## Unificación de archivos

Se unifican los 30 archivos en uno solo

In [19]:
df_list = []

columnas = [
    "Record Type",
    "Record Version",
    "Time Stamp",
    "Premise ID",
    "ESIID",
    "Provisioned",
    "Meter ID",
    "Purpose",
    "Commodity",
    "Units",
    "Calculation Constant",
    "Interval",
    "Count",
    "FirstIntervalDateTime",
    "Data"
]

for file in files:
    print("Procesando:", file)
    
    with open(file, 'r', encoding='latin-1') as f:
        rows = []
        
        for line in f:
            parts = line.strip().split("~")
            
            fixed = parts[:14]                 # columnas fijas
            data = "~~".join(parts[14:])       # intervalos
            
            rows.append(fixed + [data])
    
    temp = pd.DataFrame(rows, columns=columnas)
    df_list.append(temp)

df = pd.concat(df_list, ignore_index=True)

print("Shape:", df.shape)
print(df.head())

Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_01032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_02032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_03032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_0000_04032026.txt
Procesando: C:\Users\rroman\OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)\Documents\Unidad\2026\Capacitación\Data Science y Machine Learning\proyecto final\data\raw\intervals_000

In [20]:
df = df[df["Record Type"] != "Record Type"]
print(df["ESIID"].unique()[:10])

<StringArray>
['41-0301', '41-0300', '41-0320', '41-0321',        '', '41-0310', '41-0311',
 '41-0330', '41-0331']
Length: 9, dtype: str


## Revisión de datos

1. ¿Cuantos medidores 41-0330 y 41-0331 hay? 


In [21]:
df_indirectos = df[df["ESIID"].isin(["41-0330", "41-0331"])]

print("Registros:", df_indirectos.shape)
print("Medidores únicos:", df_indirectos["Meter ID"].nunique())

Registros: (1434844, 15)
Medidores únicos: 3732


2. ¿Cuantos de esos medidores tienen vacios en su data para eliminarlos? 


In [22]:
df_indirectos["has_N"] = df_indirectos["Data"].str.contains("~N~~", na=False)

medidores_con_N = df_indirectos[df_indirectos["has_N"]]["Meter ID"].nunique()
medidores_total = df_indirectos["Meter ID"].nunique()

print("Medidores con datos faltantes (flag N):", medidores_con_N)
print("Total medidores:", medidores_total)
print("Porcentaje:", medidores_con_N / medidores_total * 100)

Medidores con datos faltantes (flag N): 2227
Total medidores: 3732
Porcentaje: 59.67309753483387


Eliminando los medidores que contengan vacios por problemas de comunicación

In [23]:
medidores_validos = df_indirectos.groupby("Meter ID")["has_N"].max()
medidores_validos = medidores_validos[medidores_validos == False].index

df_indirectos = df_indirectos[df_indirectos["Meter ID"].isin(medidores_validos)]

3. ¿Cuales son las variables asociados a esos codigo de materiales? Se pensaría que todos traen las mismas variables, puede ser que no.

In [24]:
variables_por_material = df_indirectos.groupby("ESIID")["Units"].unique()

for k, v in variables_por_material.items():
    print(f"\nESIID {k}:")
    print(list(v))


ESIID 41-0330:
['kWh-DelLPABC', 'kWh-RcvdLPABC', 'kVARh-DelLPABC', 'kVARh-RcvdLPABC', 'CalculatedIrmsPhaseALPABC', 'CalculatedIrmsPhaseBLPABC', 'CalculatedIrmsPhaseCLPABC', 'CalculatedVrmsPhaseALPABC', 'CalculatedVrmsPhaseBLPABC', 'CalculatedVrmsPhaseCLPABC', 'VsagAnyPhaseLPABC', 'VswellAnyPhaseLPABC', 'CalculatedIrmsNeutralLPABC', 'KWH', '-KWH', 'kVARhDel', 'kVARh-Rcvd', 'CalculatedIrmsPhaseALP', 'CalculatedIrmsPhaseBLP', 'CalculatedIrmsPhaseCLP', 'CalculatedVrmsPhaseALP', 'CalculatedVrmsPhaseBLP', 'CalculatedVrmsPhaseCLP', 'Sag_V_Any_Phase', 'Swell_V_Any_Phase', 'CalculatedIrmsNeutralLP', 'IrmsPhaseALPABC', 'IrmsPhaseBLPABC', 'IrmsPhaseCLPABC', 'VrmsPhaseALPABC', 'VrmsPhaseBLPABC', 'VrmsPhaseCLPABC', 'IrmsNeutralLPABC']

ESIID 41-0331:
['KWH', '-KWH', 'kVARhDel', 'kVARh-Rcvd', 'CalculatedIrmsPhaseALP', 'CalculatedIrmsPhaseBLP', 'CalculatedIrmsPhaseCLP', 'CalculatedVrmsPhaseALP', 'CalculatedVrmsPhaseBLP', 'CalculatedVrmsPhaseCLP', 'Sag_V_Any_Phase', 'Swell_V_Any_Phase', 'CalculatedIr

In [25]:
variables_por_medidor = df_indirectos.groupby("Meter ID")["Units"].nunique()

print(variables_por_medidor.describe())

count    1505.0
mean       13.0
std         0.0
min        13.0
25%        13.0
50%        13.0
75%        13.0
max        13.0
Name: Units, dtype: float64


Creando un diccionario de variables, ya que las mismas variables fisicas se le han asignado diferentes etiquetas dependiendo del modelo del medidor

In [26]:
normalizacion = {
    # Energía
    "KWH": "kWh_Del",
    "kWh-DelLPABC": "kWh_Del",
    
    "-KWH": "kWh_Rec",
    "kWh-RcvdLPABC": "kWh_Rec",
    
    "kVARhDel": "kVARh_Del",
    "kVARh-DelLPABC": "kVARh_Del",
    
    "kVARh-Rcvd": "kVARh_Rec",
    "kVARh-RcvdLPABC": "kVARh_Rec",
    
    # Corriente
    "CalculatedIrmsPhaseALP": "CurrentPhaseA",
    "CalculatedIrmsPhaseALPABC": "CurrentPhaseA",
    "IrmsPhaseALPABC": "CurrentPhaseA",
    
    "CalculatedIrmsPhaseBLP": "CurrentPhaseB",
    "CalculatedIrmsPhaseBLPABC": "CurrentPhaseB",
    "IrmsPhaseBLPABC": "CurrentPhaseB",
    
    "CalculatedIrmsPhaseCLP": "CurrentPhaseC",
    "CalculatedIrmsPhaseCLPABC": "CurrentPhaseC",
    "IrmsPhaseCLPABC": "CurrentPhaseC",
    
    # Voltaje
    "CalculatedVrmsPhaseALP": "VoltagePhaseA",
    "CalculatedVrmsPhaseALPABC": "VoltagePhaseA",
    "VrmsPhaseALPABC": "VoltagePhaseA",
    
    "CalculatedVrmsPhaseBLP": "VoltagePhaseB",
    "CalculatedVrmsPhaseBLPABC": "VoltagePhaseB",
    "VrmsPhaseBLPABC": "VoltagePhaseB",
    
    "CalculatedVrmsPhaseCLP": "VoltagePhaseC",
    "CalculatedVrmsPhaseCLPABC": "VoltagePhaseC",
    "VrmsPhaseCLPABC": "VoltagePhaseC",
    
    # Eventos
    "Sag_V_Any_Phase": "VoltageSag",
    "VsagAnyPhaseLPABC": "VoltageSag",
    
    "Swell_V_Any_Phase": "VoltageSwell",
    "VswellAnyPhaseLPABC": "VoltageSwell",
    
    # Neutro
    "CalculatedIrmsNeutralLP": "CurrentPhaseN",
    "CalculatedIrmsNeutralLPABC": "CurrentPhaseN",
    "IrmsNeutralLPABC": "CurrentPhaseN"
}

Aplicar normalización de etiquetas

In [27]:
df_indirectos["Units_original"] = df_indirectos["Units"]

df_indirectos["Units"] = df_indirectos["Units"].str.strip()
df_indirectos["Units"] = df_indirectos["Units"].replace(normalizacion)

In [28]:
variables_interes = [
    'kWh_Del',
    'kWh_Rec',
    'kVARh_Del',
    'kVARh_Rec',
    'CurrentPhaseA',
    'CurrentPhaseB',
    'CurrentPhaseC',
    'VoltagePhaseA',
    'VoltagePhaseB',
    'VoltagePhaseC',
    'VoltageSag',
    'VoltageSwell',
    'CurrentPhaseN'
]


Filtrado de datos

In [29]:
df_filtrado = df_indirectos[df_indirectos["Units"].isin(variables_interes)]

vars_por_medidor = df_filtrado.groupby("Meter ID")["Units"].nunique()

medidores_completos = vars_por_medidor[
    vars_por_medidor == len(variables_interes)
].index

df_final = df_filtrado[df_filtrado["Meter ID"].isin(medidores_completos)]

In [30]:
df_final.groupby("Meter ID")["Units"].unique().head()

Meter ID
F-74767    [kWh_Del, kWh_Rec, kVARh_Del, kVARh_Rec, Curre...
F-78012    [kWh_Del, kWh_Rec, kVARh_Del, kVARh_Rec, Curre...
F-78014    [kWh_Del, kWh_Rec, kVARh_Del, kVARh_Rec, Curre...
F-78644    [kWh_Del, kWh_Rec, kVARh_Del, kVARh_Rec, Curre...
F-78646    [kWh_Del, kWh_Rec, kVARh_Del, kVARh_Rec, Curre...
Name: Units, dtype: object

## Extendiendo los datos

Limpiar y convertir la columna data a intervalos

In [32]:
df_final["Interval_List"] = (
    df_final["Data"]
    .str.strip("~")
    .str.split("~~")
)

Expandir los intervalos

In [39]:
intervalos = df_final["Interval_List"].apply(pd.Series)

In [40]:
df_expandido = pd.concat(
    [df_final[["Meter ID", "FirstIntervalDateTime", "Units"]], intervalos],
    axis=1
)

In [41]:
df_long = df_expandido.melt(
    id_vars=["Meter ID", "FirstIntervalDateTime", "Units"],
    var_name="Interval",
    value_name="Value"
)

In [42]:
df_long["FirstIntervalDateTime"] = pd.to_datetime(
    df_long["FirstIntervalDateTime"],
    format="%m%d%Y%I%M%S%p"
)

df_long["Datetime"] = (
    df_long["FirstIntervalDateTime"] +
    pd.to_timedelta(df_long["Interval"].astype(int) * 15, unit="m")
)

In [43]:
print(df_long[["FirstIntervalDateTime", "Interval", "Datetime"]].head(20))

   FirstIntervalDateTime Interval            Datetime
0    2026-02-28 00:15:00        0 2026-02-28 00:15:00
1    2026-02-28 00:15:00        0 2026-02-28 00:15:00
2    2026-02-28 00:15:00        0 2026-02-28 00:15:00
3    2026-02-28 00:15:00        0 2026-02-28 00:15:00
4    2026-02-28 00:15:00        0 2026-02-28 00:15:00
5    2026-02-28 00:15:00        0 2026-02-28 00:15:00
6    2026-02-28 00:15:00        0 2026-02-28 00:15:00
7    2026-02-28 00:15:00        0 2026-02-28 00:15:00
8    2026-02-28 00:15:00        0 2026-02-28 00:15:00
9    2026-02-28 00:15:00        0 2026-02-28 00:15:00
10   2026-02-28 00:15:00        0 2026-02-28 00:15:00
11   2026-02-28 00:15:00        0 2026-02-28 00:15:00
12   2026-02-28 00:15:00        0 2026-02-28 00:15:00
13   2026-02-28 00:15:00        0 2026-02-28 00:15:00
14   2026-02-28 00:15:00        0 2026-02-28 00:15:00
15   2026-02-28 00:15:00        0 2026-02-28 00:15:00
16   2026-02-28 00:15:00        0 2026-02-28 00:15:00
17   2026-02-28 00:15:00    

In [45]:
df_long["Hora"] = df_long["Datetime"].dt.strftime("%H:%M")
print(sorted(df_long["Hora"].unique()))

['00:00', '00:15', '00:30', '00:45', '01:00', '01:15', '01:30', '01:45', '02:00', '02:15', '02:30', '02:45', '03:00', '03:15', '03:30', '03:45', '04:00', '04:15', '04:30', '04:45', '05:00', '05:15', '05:30', '05:45', '06:00', '06:15', '06:30', '06:45', '07:00', '07:15', '07:30', '07:45', '08:00', '08:15', '08:30', '08:45', '09:00', '09:15', '09:30', '09:45', '10:00', '10:15', '10:30', '10:45', '11:00', '11:15', '11:30', '11:45', '12:00', '12:15', '12:30', '12:45', '13:00', '13:15', '13:30', '13:45', '14:00', '14:15', '14:30', '14:45', '15:00', '15:15', '15:30', '15:45', '16:00', '16:15', '16:30', '16:45', '17:00', '17:15', '17:30', '17:45', '18:00', '18:15', '18:30', '18:45', '19:00', '19:15', '19:30', '19:45', '20:00', '20:15', '20:30', '20:45', '21:00', '21:15', '21:30', '21:45', '22:00', '22:15', '22:30', '22:45', '23:00', '23:15', '23:30', '23:45']


In [46]:
df_dataset = df_long.pivot_table(
    index=["Meter ID", "Datetime"],
    columns="Units",
    values="Value",
    aggfunc="first"
).reset_index()

In [52]:
columnas_numericas = [
    'CurrentPhaseA',
    'CurrentPhaseB',
    'CurrentPhaseC',
    'CurrentPhaseN',
    'VoltagePhaseA',
    'VoltagePhaseB',
    'VoltagePhaseC',
    'VoltageSag',
    'VoltageSwell',
    'kVARh_Del',
    'kVARh_Rec',
    'kWh_Del',
    'kWh_Rec'
]

for col in columnas_numericas:
    df_dataset[col] = pd.to_numeric(df_dataset[col], errors="coerce")

In [53]:
print(df_dataset.dtypes)

Units
Meter ID                    str
Datetime         datetime64[us]
CurrentPhaseA           float64
CurrentPhaseB           float64
CurrentPhaseC           float64
CurrentPhaseN           float64
VoltagePhaseA           float64
VoltagePhaseB           float64
VoltagePhaseC           float64
VoltageSag              float64
VoltageSwell            float64
kVARh_Del               float64
kVARh_Rec               float64
kWh_Del                 float64
kWh_Rec                 float64
dtype: object


In [58]:
from pathlib import Path

ruta_base = Path(
    r"C:/Users/rroman/OneDrive - Empresa Eléctrica de Guatemala, S.A. (EEGSA)/Documents/Unidad/2026/Capacitación/Data Science y Machine Learning/proyecto final/data"
)

ruta_processed = ruta_base / "processed"
ruta_processed.mkdir(parents=True, exist_ok=True)

ruta_parquet = ruta_processed / "dataset_medidores.parquet"

df_dataset.to_parquet(ruta_parquet, engine="fastparquet", index=False)